# 06 – Métricas do Lote 5 (L5)

ESP-02, ESP-03, ESP-06, QA-05, ADD-05 e ADD-06, segundo `spec/metodologia_metricas.md` (secções 7E e 7F).

- Lê `processed/<sensor>_<ano>.parquet` (notebook 01), **um ficheiro de cada vez e só com as colunas necessárias**, e as tabelas `l1_cal_coverage`, `l1_cal_monthly` e `l1_cal_annual` (notebook 02).
- Por ficheiro guarda as células-dia elegíveis e o ADD-05 em `metrics/_partial_lote5/` e **retoma** se a sessão cair (`FORCE = True` refaz). Por ano junta os sensores, calcula o cell-year (dias com deteção por célula) e ESP-02/03.
- O ESP-06 precisa de todos os anos: consolida-se **por blocos de países**, lendo só as linhas desse bloco de cada ano (não carrega todos os cell-years de uma vez). Se faltar memória, reduza `ESP06_BLOCK` ou ative *High-RAM*.
- QA-05 é só um **proxy global** (dia com algum registo no sensor); não é cobertura real por região.
- Escreve `metrics/l5_*.parquet`, `metrics/_log_lote5.csv` e `metrics/_params_lote5.json`.
- O código vem do repo (`git clone` do GitHub público): `data/pipeline/metrics/`.

**Antes de correr:** o atalho `Sprinklers` tem de estar em *O Meu Drive* (ver `data/README.md`) e os notebooks 01 e 02 têm de ter corrido.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Sprinklers')        # ajustar se o atalho tiver outro nome
REPO_URL = 'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git'
REPO = Path('/content/repo')


def ensure_repo():
    if (REPO / '.git').exists():
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)


ensure_repo()
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))

In [ ]:
# Para um teste curto: SENSORS = {'modis'}; YEARS = {2000}
SENSORS = None          # None = todos
YEARS = None            # None = todos
FORCE = False           # True = refaz mesmo que o parcial ja exista
GRID_DEG = 0.1          # lado da celula, em graus (convencao do projeto)
MAX_UNKNOWN_DAYS = 0    # dias de cobertura desconhecida tolerados num mes/ano valido
ESP06_BLOCK = 40        # paises por bloco no ESP-06 (menos = menos memoria)

import subprocess
from pathlib import Path

PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
assert PROCESSED.exists(), f"Nao encontro {PROCESSED}. Corra primeiro o notebook 01 e confirme o atalho 'Sprinklers'."
REQUIRED = ['l1_cal_coverage', 'l1_cal_monthly', 'l1_cal_annual']
missing = [n for n in REQUIRED if not (METRICS / f'{n}.parquet').exists()]
if missing:
    raise FileNotFoundError(f'Faltam em {METRICS}: {missing}. Corra primeiro o notebook 02_metrics_lote1.')
PART = METRICS / '_partial_lote5'
PART.mkdir(parents=True, exist_ok=True)

In [ ]:
import json, re, time, datetime
import pandas as pd

from metrics.calendar_metrics import eligible_cell_days
from metrics.cell_recurrence import build_cell_year, esp02_active_cells, esp03_recurrence, esp06_interannual
from metrics.filter_sensitivity import add05_type_sensitivity
from metrics.coverage_steps import add06_union_step, qa05_confirmed_days

COLS = ['country', 'instrument', 'satellite', 'latitude', 'longitude', 'acq_date', 'type', 'confidence_class']
FILE_RE = re.compile(r'^(?P<sensor>modis|viirs-[a-z0-9]+)_(?P<year>\d{4})\.parquet$', re.I)


def atomic_write(df, path):
    tmp = path.with_name(path.name + '.tmp')
    df.to_parquet(tmp, index=False)
    tmp.replace(path)


def process_file(sensor, year, path):
    """Celulas-dia elegiveis e ADD-05 de um ficheiro, guardados em PART/<sensor>_<ano>/."""
    out = PART / f'{sensor}_{year}'
    out.mkdir(exist_ok=True)
    t0 = time.time()
    df = pd.read_parquet(path, columns=COLS)
    info = dict(sensor=sensor, year=year, grid_deg=GRID_DEG, n_rows=len(df))
    atomic_write(eligible_cell_days(df, GRID_DEG), out / 'cell_days.parquet')
    atomic_write(add05_type_sensitivity(df, grid_deg=GRID_DEG), out / 'add05.parquet')
    info['seconds'] = round(time.time() - t0, 1)
    (out / '_done.json').write_text(json.dumps(info))      # escrito por ultimo: marca o ficheiro como completo
    return info


def is_done(marker):
    # parcial completo E calculado com a grelha atual (senao refaz: evita misturar grelhas)
    try:
        return marker.exists() and json.loads(marker.read_text()).get('grid_deg') == GRID_DEG
    except ValueError:
        return False


def stale(marker, target):
    return (not target.exists()) or marker.stat().st_mtime > target.stat().st_mtime


def year_status(year):
    """(sensores ja processados nesse ano, series completas). `combined` so e valida se TODOS os
    ficheiros do ano estao processados; senao seria so um dos sensores com o nome errado."""
    every = [s for (s, y) in files if y == year]
    done = [s for s in every if (PART / f'{s}_{year}' / '_done.json').exists()]
    series = set()
    if any(s.startswith('modis') for s in done):
        series.add('modis')
    if any(s.startswith('viirs') for s in done):
        series.add('viirs')
    if every and len(done) == len(every):
        series.add('combined')
    return done, series

In [ ]:
files = {}
for p in sorted(PROCESSED.glob('*.parquet')):
    m = FILE_RE.match(p.name)
    if m:
        files[(m['sensor'].lower(), int(m['year']))] = p
todo = sorted(k for k in files
              if (SENSORS is None or k[0] in SENSORS) and (YEARS is None or k[1] in YEARS))
print(f"{len(files)} ficheiros em processed/, {len(todo)} selecionados")
if not todo:
    raise ValueError(f'nenhum ficheiro selecionado (SENSORS={SENSORS}, YEARS={YEARS}); '
                     f'disponiveis em processed/: {sorted(files)}')

for sensor, year in todo:
    marker = PART / f'{sensor}_{year}' / '_done.json'
    if is_done(marker) and not FORCE:
        print(f'[salta] {sensor} {year}')
        continue
    print(f'[processa] {sensor} {year}')
    print('   ', process_file(sensor, year, files[(sensor, year)]))

# por ano: junta os sensores (serie 'combined'), cell-year e ESP-02/03
for year in sorted({y for (_, y) in todo}):
    done, series = year_status(year)
    markers = [PART / f'{s}_{year}' / '_done.json' for s in done]
    targets = [PART / f'cell_year_{year}.parquet', PART / f'esp02_{year}.parquet', PART / f'esp03_{year}.parquet']
    if FORCE or any(stale(m, t) for m in markers for t in targets):
        cd = pd.concat([pd.read_parquet(m.parent / 'cell_days.parquet') for m in markers], ignore_index=True)
        cy = build_cell_year(cd)
        cy = cy[cy['series'].isin(series)].reset_index(drop=True)      # so as series completas
        if len(cy):
            atomic_write(cy, targets[0])
        else:                                                         # ano sem celulas elegiveis: sem ficheiro
            targets[0].unlink(missing_ok=True)
        atomic_write(esp02_active_cells(cy), targets[1])
        atomic_write(esp03_recurrence(cy), targets[2])
        print(f'[cell_year] {year}: {len(markers)} sensores, series {sorted(series)}, {len(cy)} linhas')

In [ ]:
dirs = sorted(d for d in PART.iterdir() if d.is_dir() and (d / '_done.json').exists())
grids = {json.loads((d / '_done.json').read_text()).get('grid_deg') for d in dirs}
assert grids <= {GRID_DEG}, f'parciais calculados com grelhas diferentes {grids}; corra com FORCE = True'


def concat_parts(pattern, keys):
    tbl = pd.concat([pd.read_parquet(p) for p in sorted(PART.glob(pattern))], ignore_index=True)
    tbl = tbl.sort_values(keys).reset_index(drop=True)
    assert not tbl.duplicated(keys).any(), f'{pattern}: chaves repetidas entre ficheiros'
    return tbl


tables = {}
tables['l5_add05_type_sensitivity'] = pd.concat(
    [pd.read_parquet(d / 'add05.parquet') for d in dirs], ignore_index=True
).sort_values(['country', 'year', 'instrument', 'variant']).reset_index(drop=True)
assert not tables['l5_add05_type_sensitivity'].duplicated(['country', 'year', 'instrument', 'variant']).any(), \
    'ADD-05 com chaves repetidas entre ficheiros (datas fora do ano do ficheiro?)'
tables['l5_esp02_active_cells'] = concat_parts('esp02_*.parquet', ['country', 'series', 'year'])
tables['l5_esp03_recurrence'] = concat_parts('esp03_*.parquet', ['country', 'series', 'year'])

# ESP-06: anos validos de l1_cal_annual; consolidacao por blocos de paises
annual = pd.read_parquet(METRICS / 'l1_cal_annual.parquet')
ok = (~annual['partial'].astype(bool)) & (annual['n_unknown_days'] <= MAX_UNKNOWN_DAYS)
# so entram (serie, ano) realmente calculados: numa execucao parcial os outros anos nao contam
computed = set()
for year in sorted({y for (_, y) in files}):
    computed |= {(s, year) for s in year_status(year)[1]}
comp = pd.DataFrame(sorted(computed), columns=['series', 'year']).astype({'series': 'object', 'year': 'int64'})
eligible_years = annual.loc[ok, ['country', 'series', 'year']].astype({'year': 'int64'}).merge(
    comp, on=['series', 'year']).reset_index(drop=True)
cy_files = sorted(PART.glob('cell_year_*.parquet'))
EMPTY_CY = pd.DataFrame({'country': [], 'series': [], 'cell': [], 'year': [], 'n_days': []}).astype(
    {'cell': 'int64', 'year': 'int64', 'n_days': 'int64'})
all_countries = sorted({c for p in cy_files for c in pd.read_parquet(p, columns=['country'])['country'].unique()}
                       | set(eligible_years['country'].unique()))     # inclui paises sem nenhuma celula elegivel
blocks = [all_countries[i:i + ESP06_BLOCK] for i in range(0, len(all_countries), ESP06_BLOCK)] or [[]]
parts06 = []
for block in blocks:
    frames = [pd.read_parquet(p, filters=[('country', 'in', block)]) for p in cy_files]
    cy = pd.concat(frames, ignore_index=True) if frames else EMPTY_CY
    parts06.append(esp06_interannual(cy, eligible_years[eligible_years['country'].isin(block)]))
    del cy, frames
tables['l5_esp06_interannual'] = pd.concat(parts06, ignore_index=True)
tables['l5_esp06_interannual'] = tables['l5_esp06_interannual'].sort_values(['country', 'series']).reset_index(drop=True)

# QA-05 e ADD-06 a partir das tabelas do L1
coverage = pd.read_parquet(METRICS / 'l1_cal_coverage.parquet')
monthly = pd.read_parquet(METRICS / 'l1_cal_monthly.parquet')
tables['l5_qa05_confirmed_days'] = qa05_confirmed_days(coverage)
tables['l5_add06_union_step'] = add06_union_step(monthly, max_unknown_days=MAX_UNKNOWN_DAYS)

for name, tbl in tables.items():
    atomic_write(tbl, METRICS / f'{name}.parquet')
    print(f'{name}: {len(tbl)} linhas')

try:
    commit = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
except Exception:
    commit = ''
params = dict(grid_deg=GRID_DEG, max_unknown_days=MAX_UNKNOWN_DAYS, esp06_block=ESP06_BLOCK,
              sensors=sorted(SENSORS) if SENSORS else None, years=sorted(YEARS) if YEARS else None,
              series_years_computed=[f'{s}:{y}' for s, y in sorted(computed)],
              repo_commit=commit, run_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'))
(METRICS / '_params_lote5.json').write_text(json.dumps(params, indent=1))
log = pd.DataFrame([dict(table=n, n_rows=len(t), **params) for n, t in tables.items()])
log.to_csv(METRICS / '_log_lote5.csv', index=False)
print(log[['table', 'n_rows']].to_string(index=False))

## Verificação
Confirma invariantes entre as tabelas e contra o L1 (falha com `AssertionError` se alguma não bater).

In [ ]:
e2, e3 = tables['l5_esp02_active_cells'], tables['l5_esp03_recurrence']
keys = ['country', 'series', 'year']
assert e2[keys + ['n_active_cells']].equals(e3[keys + ['n_active_cells']]), 'ESP-02 != n_active_cells do ESP-03'
buckets = e3[['n_cells_1d', 'n_cells_2_5d', 'n_cells_6_15d', 'n_cells_16plus']].sum(axis=1)
assert (buckets == e3['n_active_cells']).all(), 'baldes de recorrencia nao somam n_active_cells'
e6 = tables['l5_esp06_interannual']
assert (e6['n_cells_2plus_years'] <= e6['n_cells']).all()
assert e6['pct_cells_2plus'].dropna().between(0, 100).all()
q5 = tables['l5_qa05_confirmed_days']
assert q5['pct_confirmed'].between(0, 100).all() and (q5['n_days_confirmed'] <= q5['n_days_expected']).all()
a6 = tables['l5_add06_union_step']
assert (a6['ratio_combined_modis'].dropna() >= 1).all(), 'a uniao nunca pode ter menos celulas-dia que o MODIS'

# contra o L1: a soma de n_days por (pais, serie, ano) e a atividade anual do CAL-04
ann = annual.set_index(['country', 'series', 'year'])['cell_days']
checked = 0
for p in cy_files:
    year = int(re.search(r'(\d{4})', p.name).group(1))
    for (country, series), total in pd.read_parquet(p).groupby(['country', 'series'])['n_days'].sum().items():
        assert (series, year) in computed, f'{series} {year} no cell-year mas nao calculada por completo'
        assert ann.get((country, series, year)) == total, f'{country} {series} {year}: cell-year != CAL-04'
        checked += 1
print(f'OK: invariantes verificadas ({checked} comparacoes com o CAL-04)')